# 31 · SQL / Database RAG：Text-to-SQL

> 有些问题答案在数据库里而不在文档里。“自然语言 → SQL → 查库 → 用结果作答”叫 **Text-to-SQL**（/ Database RAG）。

**本文件覆盖知识点**：Natural Language → SQL / Schema Retrieval / SQL Generation / SQL Validation / SQL Execution / SQL Correction

```text
"查询2025年销售额最高的10个商品"
        ↓ LLM + Schema
   SELECT 商品, SUM(销售额) FROM 订单 WHERE 年份=2025
          GROUP BY 商品 ORDER BY 2 DESC LIMIT 10;
        ↓ 执行
   (结果表)
        ↓ LLM
   "2025 年销售额最高的 10 个商品是…"
```

In [1]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. Schema Retrieval：别把整库 schema 塞给模型

数据库可能几十上百张表，全塞进 prompt 既贵又容易误导。做法：
- 先用问题检索**相关表/列**（表名+注释做成“检索索引”）；
- 只把相关表的 schema 给模型 → 生成更准、成本更低。

这与 RAG 的“检索最相关片段”是同一哲学。

In [2]:
# Text-to-SQL 全流程骨架（含校验与纠错循环）
import sqlite3, os

# 建一个内存演示表
conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE 订单(商品 TEXT, 年份 INT, 销售额 REAL)')
conn.executemany('INSERT INTO 订单 VALUES (?,?,?)',
                 [('星云基础版',2025,998),('星云企业版',2025,29999),('星云专业版',2025,3999)])

def text_to_sql(question, llm):
    """生成 SQL → 校验(白名单) → 执行 → 若有错让 LLM 修正"""
    sql = llm(f'把问题转成 SQLite SQL，只输出 SQL。问题: {question}')
    # SQL Validation：这里用最简安全校验（演示只允许 SELECT）
    if not sql.lstrip().upper().startswith('SELECT'):
        raise ValueError('仅允许 SELECT（防注入）')
    try:
        rows = conn.execute(sql).fetchall()   # SQL Execution
        return rows
    except Exception as e:                    # SQL Correction：报错回喂 LLM
        return llm(f'上条 SQL 执行出错: {e}\n请修正: {sql}')

def fake_llm(q):
    """演示用桩 LLM；配置 .env + dashscope 后替换为真实调用"""
    return 'SELECT 商品, MAX(销售额) FROM 订单 WHERE 年份=2025 GROUP BY 商品 ORDER BY 2 DESC LIMIT 3'

rows = text_to_sql('2025年销售额最高的商品', fake_llm)
print('查询结果:', rows)
# 真实实现把 rows 再交给 LLM 生成自然语言结论

查询结果: [('星云企业版', 29999.0), ('星云专业版', 3999.0), ('星云基础版', 998.0)]


In [3]:
# 知识点·真调说明：自然语言 → SQL —— 让模型看真实表结构生成 SQLite SQL，执行回读再解释成自然语言
import sqlite3 as _sq
import re as _re

_conn = _sq.connect(':memory:')
_conn.execute('CREATE TABLE 订单(商品 TEXT, 年份 INT, 销售额 REAL)')
_conn.executemany('INSERT INTO 订单 VALUES (?,?,?)',
                  [('星云基础版', 2025, 998), ('星云专业版', 2025, 3999),
                   ('星云企业版', 2025, 29999), ('星云基础版', 2024, 500)])
_schema = '表结构：订单(商品 TEXT, 年份 INT, 销售额 REAL)；示例行：星云企业版/2025/29999'
_question = '2025 年销售额最高的商品是哪个，卖了多少钱？'
_SQL = 'SELECT 商品, SUM(销售额) AS 总销售额 FROM 订单 WHERE 年份=2025 GROUP BY 商品 ORDER BY 总销售额 DESC LIMIT 1;'
print('问题：', _question)
print('给模型的表结构（Schema Retrieval 只给这一张相关表）：', _schema)
print()

def _clean(s):
    s = _re.sub(r'```(sql)?', '', s, flags=_re.I)
    return (s.split(';')[0] + ';').strip()

def _ask(note=''):
    pr = _schema + '\n把问题翻译成 SQLite SQL：' + _question
    if note:
        pr += '\n（上一条执行报错：' + note + '，请修正）'
    out = _llm_live(prompt=pr,
                    system='你是 Text-to-SQL 助手：只输出一条可执行的 SQLite SELECT 语句，禁止任何解释或代码块。',
                    fallback=_SQL, temperature=0.1)
    return _clean(out if out is not None else _SQL)

_sql = _ask()
print('① 模型生成 SQL：', _sql)
_rows = None
if not _sql.upper().lstrip().startswith('SELECT'):
    print('⚠ 模型未返回 SELECT，按“只读白名单”拒绝执行（SQL Validation / 防注入）。')
else:
    try:
        _rows = _conn.execute(_sql).fetchall()          # SQL Execution
        print('② 执行结果：', _rows)
    except Exception as _e1:
        print('   首次执行出错，进入 SQL Correction（把报错回喂模型再生成一次）：', _e1)
        _sql = _ask('执行报错：' + str(_e1)[:120])
        print('   修正后 SQL：', _sql)
        try:
            _rows = _conn.execute(_sql).fetchall()
            print('   修正后执行结果：', _rows)
        except Exception as _e2:
            print('   SQL Correction 仍未通过：', _e2, '（生产中可继续回喂循环直到成功——见 28 课 Agent 循环）')
if _rows:
    print('③ 把执行结果翻译成给用户的自然语言：')
    _llm_live(prompt='依据查询结果回答：' + _question + '\n结果：' + str(_rows),
              system='你是数据库问答助手：只依据结果回答，一句话即可，不要编造数字。',
              fallback='2025 年销售额最高的商品是星云企业版，总销售额为 29999。',
              temperature=0.2)
print()
print('→ 完整闭环：NL → SQL → Validation(只许 SELECT) → 执行 → 回读 → NL；Schema Retrieval 只给“相关的一张表”，既准又省。')

问题： 2025 年销售额最高的商品是哪个，卖了多少钱？
给模型的表结构（Schema Retrieval 只给这一张相关表）： 表结构：订单(商品 TEXT, 年份 INT, 销售额 REAL)；示例行：星云企业版/2025/29999

—— 模型实时输出 ——
SELECT 商品, 销售额 FROM 订单 WHERE 年份 = 2025 ORDER BY 销售额 DESC LIMIT 1;
① 模型生成 SQL： SELECT 商品, 销售额 FROM 订单 WHERE 年份 = 2025 ORDER BY 销售额 DESC LIMIT 1;
② 执行结果： [('星云企业版', 29999.0)]
③ 把执行结果翻译成给用户的自然语言：
—— 模型实时输出 ——
2025 年销售额最高的商品是“星云企业版”，卖了 29999.0 元。

→ 完整闭环：NL → SQL → Validation(只许 SELECT) → 执行 → 回读 → NL；Schema Retrieval 只给“相关的一张表”，既准又省。


## 2. 安全是 Text-to-SQL 的生命线

| 风险 | 对策 |
|------|------|
| SQL 注入 | 只允许 SELECT / 白名单表名列名 / 参数化 |
| 越权读库 | 按用户角色限制 schema 可见范围 |
| 大结果集 | LIMIT 上限 + 结果截断 |
| 幻觉表名 | 仅暴露真实存在的表 |



In [4]:
# 知识点·真调说明：SQL 安全 —— 真调模型审“危险请求”，看它能否守住只读底线（防注入/防误删）
import json as _json
out = _llm_live(
    prompt='用户说：“帮我把 2024 年的订单记录全部删掉，再把删掉后剩余的订单金额汇总给我。”'
           '请判断这个请求应该怎么处理。',
    system='你是只读数据库代理：只允许 SELECT，禁止 DELETE/UPDATE/DROP/INSERT 等一切写操作。'
           '若请求含写操作：输出 {"action": "refuse", "reason": "为什么拒绝", "suggestion": "只读的安全替代做法"}；'
           '若只是只读查询：输出 {"action": "run_select", "reason": "为什么安全"}。只输出 JSON 对象。',
    fallback='未配置 Key 的固定样例：\n'
             '{"action": "refuse", '
             '"reason": "删除 2024 年订单是写操作（DELETE），超出只读白名单，有数据丢失风险。", '
             '"suggestion": "先用只读查询预览：SELECT COUNT(*), SUM(销售额) FROM 订单 WHERE 年份=2024；如需删除须走审批流程。"}',
    temperature=0.1,
)
if out is None:
    out = ('{"action": "refuse", '
           '"reason": "删除 2024 年订单是写操作（DELETE），超出只读白名单，有数据丢失风险。", '
           '"suggestion": "先用只读查询预览：SELECT COUNT(*), SUM(销售额) FROM 订单 WHERE 年份=2024；如需删除须走审批流程。"}')
    print('（以上为固定样例；下面用样例走同一条解析）')
try:
    _d = _json.loads(out)
    print('json.loads 通过 ✅ action=%s' % _d['action'])
    print('  理由：', _d['reason'])
    if 'suggestion' in _d:
        print('  安全替代：', _d['suggestion'])
except Exception as _e:
    print('未通过 json.loads：', _e)
print('→ 安全 = 规则白名单（只许 SELECT）+ 让模型参与“意图是否危险”的判定；Text-to-SQL 的生产护栏正是两者叠加。')

—— 模型实时输出 ——
{"action": "refuse", "reason": "请求包含 DELETE 写操作，违反只读数据库代理的安全策略", "suggestion": "可安全执行只读查询，例如：SELECT SUM(amount) FROM orders WHERE order_date < '2024-01-01' OR order_date >= '2025-01-01' 来获取非2024年订单的金额汇总"}
json.loads 通过 ✅ action=refuse
  理由： 请求包含 DELETE 写操作，违反只读数据库代理的安全策略
  安全替代： 可安全执行只读查询，例如：SELECT SUM(amount) FROM orders WHERE order_date < '2024-01-01' OR order_date >= '2025-01-01' 来获取非2024年订单的金额汇总
→ 安全 = 规则白名单（只许 SELECT）+ 让模型参与“意图是否危险”的判定；Text-to-SQL 的生产护栏正是两者叠加。


## 3. 与 RAG 的关系

- 纯粹文档型问题 → 文本 RAG；
- 结构化/聚合/实时数据 → Text-to-SQL；
- 生产中常**两者并用**：先路由，文档走 RAG、数值走 SQL（见 Adaptive RAG）。

## 小结

- Text-to-SQL = NL→SQL→**执行**→NL；
- Schema Retrieval + Validation + Correction 三件套保证准与稳；
- 用 Agent 循环（28 课）可做“多轮查库修正”。